# NetShield — Phase 7: Semantic Alignment & Distribution-Shift Analysis

## Purpose

This is the **decisive experiment** for the project.

The project is not trying to invent a new classifier. Its research question is whether ML-based intrusion detectors learn behavior that transfers across network environments, and whether a **common semantically aligned representation** makes that transfer more reliable.

Phase 6 established an important robustness finding: CICIDS performance is extremely high under a random split but can collapse under capture-aware evaluation. Phase 7 now focuses on the remaining central question:

> **Does the common semantic representation preserve useful detection performance while making cross-dataset behavior more interpretable and robust?**

### This notebook deliberately does NOT
- add arbitrary models,
- tune on the destination dataset,
- use `source_file` as a predictor,
- use SMOTE by default,
- delete legitimate zero values,
- claim that common features "improve" performance unless the experiment actually demonstrates it.

### Important methodological correction

UNSW-NB15 and CICIDS2017 have **no exact shared predictive feature schema**, so literal raw-feature transfer is not valid. The project therefore evaluates cross-dataset transfer through a verified semantic representation rather than pretending that differently defined columns are interchangeable.

The final conclusion may be:
- common semantic features improve transfer,
- they provide little improvement but expose the real source of degradation,
- or they do not improve transfer.

**All three are valid research outcomes.**


## Experiment map

```text
                    Common semantic representation
                               │
              ┌────────────────┼────────────────┐
              ▼                ▼                ▼
        UNSW common      CICIDS common     Distribution shift
              │                │                │
              ▼                ▼                ▼
       within-source      within-source      feature-wise
              │                │                │
              └───────┬────────┴────────┬───────┘
                      ▼                 ▼
              UNSW → CICIDS       CICIDS → UNSW
                      │                 │
                      └────────┬────────┘
                               ▼
                    transfer-gap analysis
                               │
                               ▼
                  capture/scenario sensitivity
```

### Main comparisons

1. **Native within-dataset baselines** are taken from earlier phases; this notebook does not need to retrain them.
2. **Common-feature within-dataset performance** establishes the cost of reducing each dataset to the shared semantic space.
3. **Common-feature cross-dataset transfer** is evaluated in both directions.
4. **CICIDS capture-wise transfer** tests whether a model trained on UNSW transfers equally well to different CICIDS capture scenarios.
5. **Distribution-shift diagnostics** are linked to model performance without using destination labels for tuning.
6. A final decision table determines whether the evidence supports improved robustness, no improvement, or unresolved transfer difficulty.


In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score, average_precision_score,
    roc_auc_score, confusion_matrix
)
from scipy.stats import ks_2samp

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
UNSW_ROOT = RAW_ROOT / "UNSW-NB15"
CICIDS_ROOT = RAW_ROOT / "CICIDS2017"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "semantic_alignment"
TABLE_DIR = OUTPUT_ROOT / "tables"
PLOT_DIR = OUTPUT_ROOT / "plots"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Runtime controls.
# CICIDS is large; the cap is only applied to TRAINING rows.
# Test sets remain untouched.
CICIDS_TRAIN_CAP = 600_000
RF_TREES = 100
HGB_ITER = 100

print("Project root:", PROJECT_ROOT)
print("Output root:", OUTPUT_ROOT)


In [ ]:
# Load official UNSW-NB15 train/test
unsw_train_path = UNSW_ROOT / "Training and Testing Sets" / "UNSW_NB15_training-set.csv"
unsw_test_path = UNSW_ROOT / "Training and Testing Sets" / "UNSW_NB15_testing-set.csv"

if not unsw_train_path.exists() or not unsw_test_path.exists():
    raise FileNotFoundError(
        "UNSW official train/test files were not found under "
        "data/raw/UNSW-NB15/Training and Testing Sets/"
    )

unsw_train = pd.read_csv(unsw_train_path)
unsw_test = pd.read_csv(unsw_test_path)

# Load all CICIDS CSVs and retain source_file only as metadata.
cicids_files = sorted(CICIDS_ROOT.rglob("*.csv"))
if not cicids_files:
    raise FileNotFoundError(f"No CICIDS CSV files found under {CICIDS_ROOT}")

parts = []
for path in cicids_files:
    part = pd.read_csv(path, low_memory=False)
    part.columns = part.columns.str.strip()
    part["source_file"] = path.name
    parts.append(part)

cicids = pd.concat(parts, ignore_index=True)

print("UNSW train:", unsw_train.shape)
print("UNSW test :", unsw_test.shape)
print("CICIDS    :", cicids.shape)
print("CICIDS source files:", len(cicids_files))


In [ ]:
# Binary labels: 0 = benign, 1 = attack
unsw_train = unsw_train.copy()
unsw_test = unsw_test.copy()
cicids = cicids.copy()

unsw_train["label_binary"] = pd.to_numeric(
    unsw_train["label"], errors="coerce"
).astype("int8")

unsw_test["label_binary"] = pd.to_numeric(
    unsw_test["label"], errors="coerce"
).astype("int8")

cicids["label_binary"] = (
    cicids["Label"].astype(str).str.strip().str.upper().ne("BENIGN").astype("int8")
)

print("UNSW train label proportions:")
print(unsw_train["label_binary"].value_counts(normalize=True).sort_index())

print("\nUNSW test label proportions:")
print(unsw_test["label_binary"].value_counts(normalize=True).sort_index())

print("\nCICIDS label proportions:")
print(cicids["label_binary"].value_counts(normalize=True).sort_index())


## 1. Construct the verified common semantic representation

The common representation contains only flow-level quantities whose meaning can be aligned across the two datasets.

| Common feature | UNSW-NB15 source/derivation | CICIDS2017 source/derivation |
|---|---|---|
| `flow_duration_sec` | `dur` | `Flow Duration / 1,000,000` |
| `fwd_packets` | `spkts` | `Total Fwd Packets` |
| `bwd_packets` | `dpkts` | `Total Backward Packets` |
| `fwd_bytes` | `sbytes` | `Total Length of Fwd Packets` |
| `bwd_bytes` | `dbytes` | `Total Length of Bwd Packets` |
| `total_packets` | forward + backward | forward + backward |
| `total_bytes` | forward + backward | forward + backward |
| `packet_rate` | total packets / duration | `Flow Packets/s` |
| `fwd_packet_rate` | forward packets / duration | `Fwd Packets/s` |
| `bwd_packet_rate` | backward packets / duration | `Bwd Packets/s` |
| `avg_packet_size` | total bytes / total packets | `Average Packet Size` |

**Important:** UNSW `sload`/`dload` are not substituted for CICIDS packet-rate fields. The directional packet rates are derived from packet counts and duration on UNSW, avoiding a semantic/unit mismatch.


In [ ]:
COMMON_FEATURES = [
    "flow_duration_sec", "fwd_packets", "bwd_packets",
    "fwd_bytes", "bwd_bytes", "total_packets", "total_bytes",
    "packet_rate", "fwd_packet_rate", "bwd_packet_rate",
    "avg_packet_size"
]

def safe_divide(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    denominator = denominator.replace(0, np.nan)
    return numerator.div(denominator)

def build_unsw_common(df):
    out = pd.DataFrame(index=df.index)
    out["flow_duration_sec"] = pd.to_numeric(df["dur"], errors="coerce")
    out["fwd_packets"] = pd.to_numeric(df["spkts"], errors="coerce")
    out["bwd_packets"] = pd.to_numeric(df["dpkts"], errors="coerce")
    out["fwd_bytes"] = pd.to_numeric(df["sbytes"], errors="coerce")
    out["bwd_bytes"] = pd.to_numeric(df["dbytes"], errors="coerce")
    out["total_packets"] = out["fwd_packets"] + out["bwd_packets"]
    out["total_bytes"] = out["fwd_bytes"] + out["bwd_bytes"]
    out["packet_rate"] = safe_divide(out["total_packets"], out["flow_duration_sec"])
    out["fwd_packet_rate"] = safe_divide(out["fwd_packets"], out["flow_duration_sec"])
    out["bwd_packet_rate"] = safe_divide(out["bwd_packets"], out["flow_duration_sec"])
    out["avg_packet_size"] = safe_divide(out["total_bytes"], out["total_packets"])
    return out

def build_cicids_common(df):
    out = pd.DataFrame(index=df.index)
    # CICIDS Flow Duration is represented in microseconds.
    out["flow_duration_sec"] = pd.to_numeric(
        df["Flow Duration"], errors="coerce"
    ).div(1_000_000)
    out["fwd_packets"] = pd.to_numeric(
        df["Total Fwd Packets"], errors="coerce"
    )
    out["bwd_packets"] = pd.to_numeric(
        df["Total Backward Packets"], errors="coerce"
    )
    out["fwd_bytes"] = pd.to_numeric(
        df["Total Length of Fwd Packets"], errors="coerce"
    )
    out["bwd_bytes"] = pd.to_numeric(
        df["Total Length of Bwd Packets"], errors="coerce"
    )
    out["total_packets"] = out["fwd_packets"] + out["bwd_packets"]
    out["total_bytes"] = out["fwd_bytes"] + out["bwd_bytes"]
    out["packet_rate"] = pd.to_numeric(
        df["Flow Packets/s"], errors="coerce"
    )
    out["fwd_packet_rate"] = pd.to_numeric(
        df["Fwd Packets/s"], errors="coerce"
    )
    out["bwd_packet_rate"] = pd.to_numeric(
        df["Bwd Packets/s"], errors="coerce"
    )
    out["avg_packet_size"] = pd.to_numeric(
        df["Average Packet Size"], errors="coerce"
    )
    return out

unsw_common_train = build_unsw_common(unsw_train)
unsw_common_test = build_unsw_common(unsw_test)
cicids_common = build_cicids_common(cicids)

# Remove non-finite values. Preserve legitimate zeros.
for df in [unsw_common_train, unsw_common_test, cicids_common]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Common traffic quantities should be non-negative.
for df in [unsw_common_train, unsw_common_test, cicids_common]:
    for col in COMMON_FEATURES:
        df.loc[df[col] < 0, col] = np.nan

print("Common feature matrix shapes:")
print("UNSW train:", unsw_common_train.shape)
print("UNSW test :", unsw_common_test.shape)
print("CICIDS    :", cicids_common.shape)


In [ ]:
# Audit the common representation before modeling.
quality_rows = []

for dataset_name, df in [
    ("UNSW_train", unsw_common_train),
    ("UNSW_test", unsw_common_test),
    ("CICIDS", cicids_common),
]:
    for feature in COMMON_FEATURES:
        s = df[feature]
        quality_rows.append({
            "dataset": dataset_name,
            "feature": feature,
            "missing": int(s.isna().sum()),
            "missing_pct": float(s.isna().mean() * 100),
            "zero_pct": float((s == 0).mean() * 100),
            "finite": int(np.isfinite(s.dropna()).sum()),
        })

common_quality = pd.DataFrame(quality_rows)
display(common_quality)
common_quality.to_csv(TABLE_DIR / "01_common_feature_quality.csv", index=False)


In [ ]:
# Training-only median artifacts.
# These are diagnostic/audit artifacts; sklearn pipelines still fit their
# own imputer only on the training data used in each experiment.

unsw_train_medians = unsw_common_train[COMMON_FEATURES].median()
cicids_train_medians_full = cicids_common[COMMON_FEATURES].median()

median_artifacts = pd.DataFrame({
    "feature": COMMON_FEATURES,
    "UNSW_train_median": unsw_train_medians.values,
    "CICIDS_full_median_diagnostic": cicids_train_medians_full.values
})

display(median_artifacts)
median_artifacts.to_csv(
    TABLE_DIR / "02_common_feature_median_artifacts.csv",
    index=False
)


In [ ]:
# Model definitions: the same four model families used in Phase 5.
# No hyperparameter tuning is performed here.

models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000, random_state=RANDOM_STATE
        ))
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(
            random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=RF_TREES,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(
            max_iter=HGB_ITER,
            random_state=RANDOM_STATE
        ))
    ])
}

print(list(models))


In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test,
                    experiment, model_name, source_rows=None, test_rows=None):
    fitted = clone(model)
    fitted.fit(X_train, y_train)

    y_pred = fitted.predict(X_test)

    if hasattr(fitted, "predict_proba"):
        y_score = fitted.predict_proba(X_test)[:, 1]
    else:
        y_score = None

    tn, fp, fn, tp = confusion_matrix(
        y_test, y_pred, labels=[0, 1]
    ).ravel()

    return {
        "experiment": experiment,
        "model": model_name,
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "macro_f1": f1_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "pr_auc": (
            average_precision_score(y_test, y_score)
            if y_score is not None else np.nan
        ),
        "roc_auc": (
            roc_auc_score(y_test, y_score)
            if y_score is not None and len(np.unique(y_test)) == 2
            else np.nan
        ),
        "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
        "fnr": fn / (fn + tp) if (fn + tp) else np.nan,
        "predicted_attack_rate": float(np.mean(y_pred)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "source_rows": source_rows,
        "test_rows": test_rows
    }, fitted, y_pred, y_score


## 2. Common-feature within-dataset baselines

These are necessary before judging transfer.

If a dataset loses substantial performance **even when trained and tested in the same dataset** after compression to the 11 common features, then a poor cross-dataset score cannot be attributed only to domain shift.

This creates a clean decomposition:

> native within-dataset performance → common-space within-dataset performance → common-space cross-dataset performance.


In [ ]:
# UNSW common within-dataset: official train -> official test.
X_u_train = unsw_common_train[COMMON_FEATURES]
y_u_train = unsw_train["label_binary"].astype("int8")
X_u_test = unsw_common_test[COMMON_FEATURES]
y_u_test = unsw_test["label_binary"].astype("int8")

unsw_common_rows = []
unsw_common_fitted = {}

for name, model in models.items():
    row, fitted, _, _ = evaluate_model(
        model, X_u_train, y_u_train, X_u_test, y_u_test,
        "UNSW_common_within", name,
        source_rows=len(X_u_train), test_rows=len(X_u_test)
    )
    unsw_common_rows.append(row)
    unsw_common_fitted[name] = fitted

unsw_common_results = pd.DataFrame(unsw_common_rows)
display(unsw_common_results.sort_values("f1", ascending=False))
unsw_common_results.to_csv(
    TABLE_DIR / "03_unsw_common_within_results.csv", index=False
)


In [ ]:
# CICIDS common within-dataset: reproducible stratified 80/20 split.
cicids_idx = np.arange(len(cicids_common))
cicids_y = cicids["label_binary"].to_numpy()

c_train_idx, c_test_idx = train_test_split(
    cicids_idx,
    test_size=0.20,
    stratify=cicids_y,
    random_state=RANDOM_STATE
)

# Cap only the expensive training side.
rng = np.random.default_rng(RANDOM_STATE)
if len(c_train_idx) > CICIDS_TRAIN_CAP:
    selected_train_idx = rng.choice(
        c_train_idx, size=CICIDS_TRAIN_CAP, replace=False
    )
else:
    selected_train_idx = c_train_idx

X_c_train = cicids_common.iloc[selected_train_idx][COMMON_FEATURES]
y_c_train = cicids_y[selected_train_idx]
X_c_test = cicids_common.iloc[c_test_idx][COMMON_FEATURES]
y_c_test = cicids_y[c_test_idx]

cicids_common_rows = []
cicids_common_fitted = {}

for name, model in models.items():
    row, fitted, _, _ = evaluate_model(
        model, X_c_train, y_c_train, X_c_test, y_c_test,
        "CICIDS_common_within", name,
        source_rows=len(X_c_train), test_rows=len(X_c_test)
    )
    cicids_common_rows.append(row)
    cicids_common_fitted[name] = fitted

cicids_common_results = pd.DataFrame(cicids_common_rows)
display(cicids_common_results.sort_values("f1", ascending=False))
cicids_common_results.to_csv(
    TABLE_DIR / "04_cicids_common_within_results.csv", index=False
)

print("CICIDS training rows used:", len(selected_train_idx))
print("CICIDS test rows used:", len(c_test_idx))


## 3. Common-feature cross-dataset transfer

The destination dataset is **only used for evaluation**.

### Direction A
**UNSW → CICIDS**

### Direction B
**CICIDS → UNSW**

No destination-derived hyperparameter tuning is performed.


In [ ]:
# UNSW -> CICIDS common semantic transfer.
X_u_source = unsw_common_train[COMMON_FEATURES]
y_u_source = unsw_train["label_binary"].astype("int8")

X_c_destination = cicids_common[COMMON_FEATURES]
y_c_destination = cicids["label_binary"].astype("int8")

u2c_rows = []
u2c_fitted = {}

for name, model in models.items():
    row, fitted, _, _ = evaluate_model(
        model, X_u_source, y_u_source,
        X_c_destination, y_c_destination,
        "UNSW_to_CICIDS_common", name,
        source_rows=len(X_u_source), test_rows=len(X_c_destination)
    )
    u2c_rows.append(row)
    u2c_fitted[name] = fitted

u2c_results = pd.DataFrame(u2c_rows)
display(u2c_results.sort_values("f1", ascending=False))
u2c_results.to_csv(
    TABLE_DIR / "05_unsw_to_cicids_common_results.csv", index=False
)


In [ ]:
# CICIDS -> UNSW common semantic transfer.
# Use the same capped CICIDS source training sample as the CICIDS within experiment.

X_c_source = cicids_common.iloc[selected_train_idx][COMMON_FEATURES]
y_c_source = cicids_y[selected_train_idx]

X_u_destination = unsw_common_test[COMMON_FEATURES]
y_u_destination = unsw_test["label_binary"].astype("int8")

c2u_rows = []
c2u_fitted = {}

for name, model in models.items():
    row, fitted, _, _ = evaluate_model(
        model, X_c_source, y_c_source,
        X_u_destination, y_u_destination,
        "CICIDS_to_UNSW_common", name,
        source_rows=len(X_c_source), test_rows=len(X_u_destination)
    )
    c2u_rows.append(row)
    c2u_fitted[name] = fitted

c2u_results = pd.DataFrame(c2u_rows)
display(c2u_results.sort_values("f1", ascending=False))
c2u_results.to_csv(
    TABLE_DIR / "06_cicids_to_unsw_common_results.csv", index=False
)


## 4. Transfer-gap decomposition

For each model we calculate:

- **common-space within F1**
- **common-space cross F1**
- **transfer gap = within F1 − cross F1**

This is deliberately different from comparing only raw F1 values. It asks whether the model already loses information when forced into the common representation, and then how much additional loss occurs when the environment changes.


In [ ]:
gap_rows = []

for model_name in models:
    u_within = unsw_common_results.set_index("model").loc[
        model_name, "f1"
    ]
    u2c = u2c_results.set_index("model").loc[
        model_name, "f1"
    ]

    c_within = cicids_common_results.set_index("model").loc[
        model_name, "f1"
    ]
    c2u = c2u_results.set_index("model").loc[
        model_name, "f1"
    ]

    gap_rows.append({
        "model": model_name,
        "direction": "UNSW_to_CICIDS",
        "common_within_f1": u_within,
        "common_cross_f1": u2c,
        "transfer_gap": u_within - u2c,
        "relative_retention": u2c / u_within if u_within else np.nan
    })

    gap_rows.append({
        "model": model_name,
        "direction": "CICIDS_to_UNSW",
        "common_within_f1": c_within,
        "common_cross_f1": c2u,
        "transfer_gap": c_within - c2u,
        "relative_retention": c2u / c_within if c_within else np.nan
    })

transfer_gaps = pd.DataFrame(gap_rows)
display(transfer_gaps)
transfer_gaps.to_csv(
    TABLE_DIR / "07_common_representation_transfer_gaps.csv",
    index=False
)


## 5. Capture-wise UNSW → CICIDS transfer

Phase 6 showed that CICIDS performance changes dramatically across capture scenarios.

We now ask a more specific question:

> **When the model is trained on UNSW using common semantic features, are some CICIDS capture scenarios substantially harder to transfer to than others?**

This is a stronger test of dataset-dependent behavior than a single pooled CICIDS score.

`source_file` is used **only to group evaluation rows**, never as a predictive feature.


In [ ]:
capture_rows = []
capture_predictions = {}

for source_name, group in cicids.groupby("source_file", sort=True):
    test_idx = group.index.to_numpy()

    X_capture = cicids_common.loc[test_idx, COMMON_FEATURES]
    y_capture = cicids.loc[test_idx, "label_binary"].astype("int8")

    for model_name, fitted in u2c_fitted.items():
        y_pred = fitted.predict(X_capture)

        if hasattr(fitted, "predict_proba"):
            y_score = fitted.predict_proba(X_capture)[:, 1]
        else:
            y_score = None

        tn, fp, fn, tp = confusion_matrix(
            y_capture, y_pred, labels=[0, 1]
        ).ravel()

        capture_rows.append({
            "source_file": source_name,
            "model": model_name,
            "rows": len(test_idx),
            "attack_rate": float(y_capture.mean()),
            "precision": precision_score(y_capture, y_pred, zero_division=0),
            "recall": recall_score(y_capture, y_pred, zero_division=0),
            "f1": f1_score(y_capture, y_pred, zero_division=0),
            "macro_f1": f1_score(
                y_capture, y_pred, average="macro", zero_division=0
            ),
            "pr_auc": (
                average_precision_score(y_capture, y_score)
                if y_score is not None and y_capture.nunique() == 2
                else np.nan
            ),
            "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
            "fnr": fn / (fn + tp) if (fn + tp) else np.nan,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
        })

capture_transfer_results = pd.DataFrame(capture_rows)

display(
    capture_transfer_results.sort_values(
        ["source_file", "f1"], ascending=[True, False]
    )
)

capture_transfer_results.to_csv(
    TABLE_DIR / "08_unsw_to_cicids_capture_wise_transfer.csv",
    index=False
)


In [ ]:
# Identify the hardest capture scenario for each model.
hardest_capture = (
    capture_transfer_results
    .sort_values(["model", "f1"])
    .groupby("model", as_index=False)
    .first()
)

display(hardest_capture[
    ["model", "source_file", "rows", "attack_rate",
     "precision", "recall", "f1", "fpr", "fnr"]
])

hardest_capture.to_csv(
    TABLE_DIR / "09_hardest_cicids_capture_by_model.csv",
    index=False
)


## 6. Feature-distribution shift

We now quantify the same semantic features that the models are using.

For each feature we compute:

- Kolmogorov–Smirnov statistic
- median in UNSW and CICIDS
- robust median ratio
- zero-rate difference

The KS statistic is treated as a **distribution-shift diagnostic**, not a feature-selection score. With millions of observations, tiny differences can become statistically significant, so practical effect size matters.

No destination labels are used in this analysis.


In [ ]:
shift_rows = []

for feature in COMMON_FEATURES:
    u = unsw_common_train[feature].dropna()
    c = cicids_common[feature].dropna()

    if len(u) == 0 or len(c) == 0:
        continue

    ks_stat = ks_2samp(u, c).statistic

    u_median = float(u.median())
    c_median = float(c.median())

    # Robust ratio using +epsilon to avoid division by zero.
    eps = 1e-12
    median_ratio = (
        max(u_median, eps) / max(c_median, eps)
    )

    shift_rows.append({
        "feature": feature,
        "unsw_median": u_median,
        "cicids_median": c_median,
        "median_ratio_unsw_to_cicids": median_ratio,
        "abs_log2_median_ratio": abs(np.log2(median_ratio)),
        "unsw_zero_pct": float((u == 0).mean() * 100),
        "cicids_zero_pct": float((c == 0).mean() * 100),
        "absolute_zero_rate_difference_pct":
            abs(float((u == 0).mean() - (c == 0).mean()) * 100),
        "ks_statistic": float(ks_stat),
        "unsw_n": len(u),
        "cicids_n": len(c)
    })

distribution_shift = pd.DataFrame(shift_rows).sort_values(
    ["ks_statistic", "abs_log2_median_ratio"],
    ascending=False
)

display(distribution_shift)
distribution_shift.to_csv(
    TABLE_DIR / "10_common_feature_distribution_shift.csv",
    index=False
)


## 7. Link feature shift to transfer difficulty

A single global F1 score cannot prove causation.

Instead, this section produces a **descriptive evidence table** that connects:

- feature distribution mismatch,
- capture-wise transfer performance,
- and overall directional transfer.

The result is explicitly labelled as **associational evidence**, not causal proof.


In [ ]:
# Capture-level feature shift: compare each CICIDS source file with UNSW.
capture_shift_rows = []

for source_name, group in cicids.groupby("source_file", sort=True):
    idxs = group.index
    for feature in COMMON_FEATURES:
        u = unsw_common_train[feature].dropna()
        c = cicids_common.loc[idxs, feature].dropna()

        if len(u) == 0 or len(c) == 0:
            continue

        capture_shift_rows.append({
            "source_file": source_name,
            "feature": feature,
            "ks_statistic": float(ks_2samp(u, c).statistic),
            "unsw_median": float(u.median()),
            "capture_median": float(c.median()),
            "abs_log2_median_ratio": abs(
                np.log2(
                    max(float(u.median()), 1e-12) /
                    max(float(c.median()), 1e-12)
                )
            )
        })

capture_shift = pd.DataFrame(capture_shift_rows)
capture_shift.to_csv(
    TABLE_DIR / "11_capture_level_feature_shift.csv",
    index=False
)

# Merge average feature-shift diagnostics with model-level capture F1.
capture_shift_summary = (
    capture_shift.groupby("source_file", as_index=False)
    .agg(
        mean_ks=("ks_statistic", "mean"),
        max_ks=("ks_statistic", "max"),
        mean_abs_log2_median_ratio=("abs_log2_median_ratio", "mean")
    )
)

capture_f1_summary = (
    capture_transfer_results.groupby("source_file", as_index=False)
    .agg(
        mean_f1=("f1", "mean"),
        min_f1=("f1", "min"),
        mean_fpr=("fpr", "mean"),
        mean_fnr=("fnr", "mean")
    )
)

capture_shift_vs_performance = capture_shift_summary.merge(
    capture_f1_summary,
    on="source_file",
    how="left"
)

display(
    capture_shift_vs_performance.sort_values("mean_f1")
)

capture_shift_vs_performance.to_csv(
    TABLE_DIR / "12_capture_shift_vs_transfer_performance.csv",
    index=False
)


## 8. Representation decision rule

This is an **interpretation aid**, not a hidden optimization procedure.

We classify the common representation using the evidence:

- **Potential improvement**: cross-dataset performance is stronger than the project's earlier common-transfer baseline and retains useful recall/F1 without unacceptable FPR.
- **No clear improvement**: transfer remains poor despite semantic alignment.
- **Mixed**: one direction/model family benefits while the other does not.

Because literal raw-feature cross-dataset transfer is impossible with the observed schemas, we do **not** claim a direct raw-vs-common cross-dataset improvement. Instead, we report whether semantic alignment reduces transfer degradation or mainly makes the failure more interpretable.


In [ ]:
# Consolidated result table.
summary_rows = []

for model_name in models:
    u_within = unsw_common_results.set_index("model").loc[model_name]
    c_within = cicids_common_results.set_index("model").loc[model_name]
    u2c = u2c_results.set_index("model").loc[model_name]
    c2u = c2u_results.set_index("model").loc[model_name]

    summary_rows.append({
        "model": model_name,
        "UNSW_common_within_f1": u_within["f1"],
        "UNSW_to_CICIDS_f1": u2c["f1"],
        "UNSW_to_CICIDS_recall": u2c["recall"],
        "UNSW_to_CICIDS_fpr": u2c["fpr"],
        "UNSW_to_CICIDS_transfer_gap": u_within["f1"] - u2c["f1"],
        "CICIDS_common_within_f1": c_within["f1"],
        "CICIDS_to_UNSW_f1": c2u["f1"],
        "CICIDS_to_UNSW_recall": c2u["recall"],
        "CICIDS_to_UNSW_fpr": c2u["fpr"],
        "CICIDS_to_UNSW_transfer_gap": c_within["f1"] - c2u["f1"],
    })

representation_summary = pd.DataFrame(summary_rows)
display(representation_summary)

representation_summary.to_csv(
    TABLE_DIR / "13_semantic_alignment_master_summary.csv",
    index=False
)


## 9. Visualizations for the paper/report

The plots are designed around the research questions rather than generic model dashboards.


In [ ]:
# Plot 1: common-space within vs cross F1.
plot_df = transfer_gaps.copy()

models_order = list(models.keys())
x = np.arange(len(models_order))
width = 0.18

fig, ax = plt.subplots(figsize=(12, 6))

for j, direction in enumerate(["UNSW_to_CICIDS", "CICIDS_to_UNSW"]):
    subset = plot_df[plot_df["direction"] == direction].set_index("model")
    within_vals = [subset.loc[m, "common_within_f1"] for m in models_order]
    cross_vals = [subset.loc[m, "common_cross_f1"] for m in models_order]

    offset = (j - 0.5) * width
    ax.bar(x + offset, within_vals, width=width/2, label=f"{direction}: within")
    ax.bar(x + offset + width/2, cross_vals, width=width/2, label=f"{direction}: cross")

ax.set_xticks(x)
ax.set_xticklabels(models_order, rotation=20, ha="right")
ax.set_ylabel("F1")
ax.set_title("Common Semantic Representation: Within vs Cross-Dataset F1")
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / "01_common_within_vs_cross_f1.png", dpi=180)
plt.show()


In [ ]:
# Plot 2: capture-wise UNSW -> CICIDS transfer.
fig, ax = plt.subplots(figsize=(14, 7))

pivot = capture_transfer_results.pivot(
    index="source_file", columns="model", values="f1"
)

pivot.plot(kind="bar", ax=ax)
ax.set_ylabel("F1")
ax.set_xlabel("CICIDS capture/source file")
ax.set_title("UNSW → CICIDS Common-Feature Transfer by Capture Scenario")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="Model")
fig.tight_layout()
fig.savefig(PLOT_DIR / "02_capture_wise_transfer_f1.png", dpi=180)
plt.show()


In [ ]:
# Plot 3: distribution shift ranking.
top_shift = distribution_shift.sort_values(
    "ks_statistic", ascending=False
).head(11)

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(
    top_shift["feature"][::-1],
    top_shift["ks_statistic"][::-1]
)
ax.set_xlabel("KS statistic")
ax.set_title("Distribution Shift Across Common Semantic Features")
fig.tight_layout()
fig.savefig(PLOT_DIR / "03_common_feature_distribution_shift.png", dpi=180)
plt.show()


## 10. Final interpretation report

This cell creates a compact machine-readable conclusion so the evidence can be incorporated into the mid-semester report and final paper without manually copying notebook outputs.


In [ ]:
# Generate a conservative, evidence-based interpretation summary.
best_u2c = u2c_results.loc[u2c_results["f1"].idxmax()]
best_c2u = c2u_results.loc[c2u_results["f1"].idxmax()]

mean_u2c_gap = transfer_gaps.loc[
    transfer_gaps["direction"] == "UNSW_to_CICIDS", "transfer_gap"
].mean()

mean_c2u_gap = transfer_gaps.loc[
    transfer_gaps["direction"] == "CICIDS_to_UNSW", "transfer_gap"
].mean()

hardest_capture = capture_transfer_results.loc[
    capture_transfer_results["f1"].idxmin()
]

interpretation = {
    "project": "NetShield",
    "phase": "07_semantic_alignment_and_shift_analysis",
    "research_question": (
        "Whether a common semantically aligned representation "
        "improves or clarifies cross-dataset robustness."
    ),
    "common_feature_count": len(COMMON_FEATURES),
    "best_UNSW_to_CICIDS_model": best_u2c["model"],
    "best_UNSW_to_CICIDS_f1": float(best_u2c["f1"]),
    "best_CICIDS_to_UNSW_model": best_c2u["model"],
    "best_CICIDS_to_UNSW_f1": float(best_c2u["f1"]),
    "mean_UNSW_to_CICIDS_common_transfer_gap": float(mean_u2c_gap),
    "mean_CICIDS_to_UNSW_common_transfer_gap": float(mean_c2u_gap),
    "hardest_capture_for_common_UNSW_to_CICIDS": hardest_capture["source_file"],
    "hardest_capture_model": hardest_capture["model"],
    "hardest_capture_f1": float(hardest_capture["f1"]),
    "important_caution": (
        "No direct raw-feature cross-dataset comparison is claimed because "
        "the observed predictive schemas do not share an exact common feature set."
    ),
    "interpretation_rule": (
        "Common semantic alignment is judged by transfer performance, "
        "within-to-cross degradation, recall/FPR trade-offs, and capture-wise robustness; "
        "not by a single accuracy score."
    )
}

with open(OUTPUT_ROOT / "phase7_interpretation.json", "w", encoding="utf-8") as f:
    json.dump(interpretation, f, indent=2)

print(json.dumps(interpretation, indent=2))


In [ ]:
# Final audit manifest.
manifest = {
    "project": "NetShield",
    "phase": "07_semantic_alignment_and_shift_analysis",
    "random_state": RANDOM_STATE,
    "models": list(models.keys()),
    "common_features": COMMON_FEATURES,
    "cicids_training_cap": CICIDS_TRAIN_CAP,
    "rf_trees": RF_TREES,
    "hist_gradient_boosting_iterations": HGB_ITER,
    "primary_metrics": [
        "precision", "recall", "f1", "macro_f1",
        "pr_auc", "fpr", "fnr", "confusion_matrix"
    ],
    "experiments": [
        "UNSW_common_within",
        "CICIDS_common_within",
        "UNSW_to_CICIDS_common",
        "CICIDS_to_UNSW_common",
        "UNSW_to_CICIDS_capture_wise"
    ],
    "destination_tuning": False,
    "source_file_as_predictor": False,
    "zeros_preserved": True,
    "inf_to_nan": True,
    "common_representation_unit_alignment": {
        "CICIDS_Flow_Duration": "microseconds converted to seconds",
        "UNSW_directional_packet_rates": "derived from packet counts / duration"
    },
    "notes": [
        "Literal raw-feature cross-dataset transfer is not claimed because the datasets have no exact shared predictive schema.",
        "Common semantic features are evaluated as a controlled representation, not as a guarantee of improved performance.",
        "Capture-wise analysis uses source_file only for grouping evaluation rows.",
        "No destination dataset was used for hyperparameter tuning."
    ]
}

with open(OUTPUT_ROOT / "phase7_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Phase 7 notebook setup complete.")
print("Results will be saved under:", OUTPUT_ROOT)
